# Aim 3 — LightGBM 5-class staging baseline (T5)

Runs `scripts/fit_aim3_staging_baseline.py` on the full SHHS-1 cohort: EEG-only, multimodal and cardiorespiratory-only configs. Each config is evaluated on the trimmed window (30 min either side of the sleep period) **and** on the whole night.

**Runtime:** CPU with **High-RAM** (the multimodal matrix is ~8 GB; peak ~11–12 GB, which does not fit the standard 12.7 GB runtime). No GPU needed.

**Expected time:** roughly 2–3 h for cell 5 on a High-RAM VM (multimodal ~1 h, cardioresp_only ~45 min, eeg_only ~15 min, plus loading). Results and the log are written **straight to Drive** as each config finishes, so a disconnect loses at most the config in progress.

**Before running:** the script must be pushed to GitHub (cell 2 checks, and records the commit). Features come from the same Drive tar the May recovery run used (`features-phase1batch-v1`).

Vault note: `Thesis Vault/Experiments/2026-09-29 - Aim 3 LightGBM staging baseline.md`.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, time, json, subprocess, pathlib
DRIVE_ROOT = '/content/drive/MyDrive/Thesis'
assert os.path.exists(DRIVE_ROOT), f'Expected {DRIVE_ROOT} to exist'
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
print('Drive root OK:', DRIVE_ROOT)

In [ ]:
# 2. Pull repo from GitHub, confirm the Aim 3 script is present, record the commit
from google.colab import userdata
PAT = userdata.get('GITHUB_PAT_THESIS')
REPO = 'jameswudo1019hack/thesis-bme-pipeline'
if pathlib.Path('/content/Code/.git').exists():
    subprocess.run(['git', '-C', '/content/Code', 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth=50', f'https://{PAT}@github.com/{REPO}.git', '/content/Code'], check=True)
SHA = subprocess.run(['git', '-C', '/content/Code', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
subprocess.run(['git', '-C', '/content/Code', 'log', '-1', '--oneline'], check=True)
assert pathlib.Path('/content/Code/scripts/fit_aim3_staging_baseline.py').exists(), \
    'fit_aim3_staging_baseline.py not on GitHub yet: commit and push it from the Mac first'
print('Aim 3 script present at commit', SHA)

In [ ]:
# 3. Extract the features tar from Drive (skipped if already extracted this session)
TAR = f'{DRIVE_ROOT}/data/features-phase1batch-v1.tar'
DST = '/content/Code/features'
assert os.path.exists(TAR), f'Tar not found at {TAR}'
n = len([f for f in os.listdir(DST) if f.endswith('.parquet') and not f.startswith('._')]) if os.path.exists(DST) else 0
if n < 5793:
    t0 = time.time()
    subprocess.run(['tar', '-xf', TAR, '-C', '/content/Code'], check=True)
    n = len([f for f in os.listdir(DST) if f.endswith('.parquet') and not f.startswith('._')])
    print(f'Extracted in {time.time()-t0:.0f}s')
print('parquets:', n)
assert n >= 5793, 'expected at least 5,793 subject parquets'

In [ ]:
# 4. Dependencies + runtime check (High-RAM required)
!pip install -q lightgbm pyarrow click scikit-learn 2>&1 | tail -2
import multiprocessing, psutil
ram_gb = psutil.virtual_memory().total / 1e9
print('CPUs:', multiprocessing.cpu_count(), '| RAM GB:', round(ram_gb, 1))
assert ram_gb > 20, 'Switch to a High-RAM runtime: Runtime > Change runtime type > High-RAM'

In [ ]:
# 5. Run all three configs, writing straight to a timestamped Drive folder.
#    pipefail makes the cell's exit code the script's, not grep's; the assert stops Run-all on failure.
%cd /content/Code
REF = f'{DRIVE_ROOT}/results/aim2_v6_past_only/test_predictions.parquet'
assert os.path.exists(REF), f'Aim 2 reference predictions not found at {REF}'
STAMP = time.strftime('%Y%m%d-%H%M')
OUT = f'{DRIVE_ROOT}/results/aim3_staging_v1_{STAMP}'
LOG = f'{DRIVE_ROOT}/logs/aim3_staging_{STAMP}.log'
print('writing to', OUT)
!bash -o pipefail -c 'python3 -u scripts/fit_aim3_staging_baseline.py --configs eeg_only,multimodal,cardioresp_only --aim2-reference "{REF}" --out-root "{OUT}" 2>&1 | tee "{LOG}" | grep -v "^\[LightGBM\]"'
assert _exit_code == 0, f'fit script failed or was killed (exit {_exit_code}; 137 = SIGKILL, usually out of memory). Log: {LOG}'

In [ ]:
# 6. Check the run is complete on Drive (nothing is deleted or overwritten)
EXPECTED = {'eeg_only', 'multimodal', 'cardioresp_only'}
for c in sorted(EXPECTED):
    assert os.path.exists(f'{OUT}/{c}/metrics.json'), f'{c} did not finish - see {LOG}'
s = json.load(open(f'{OUT}/summary.json'))
assert set(s['configs']) == EXPECTED, f"incomplete run: {list(s['configs'])}"
print('complete; results on Drive ->', OUT, '| commit', s['env'].get('git_sha'))

In [ ]:
# 7. Summary
print(f"{'config':16s} {'feat':>5s} {'acc':>7s} {'kappa':>7s} {'mF1':>7s}   W     N1    N2    N3    REM   kappa CI              whole-night kappa")
for c, m in s['configs'].items():
    t = m['test']; ci = m['test_ci95_subject_bootstrap']['kappa']; w = m['whole_night']['test']
    print(f"{c:16s} {m['n_features']:>5d} {t['accuracy']:>7.4f} {t['kappa']:>7.4f} {t['macro_f1']:>7.4f}  "
          + ' '.join(f"{t['f1'][k]:.3f}" for k in ['W','N1','N2','N3','REM'])
          + f"   [{ci[0]:.4f}, {ci[1]:.4f}]   {w['kappa']:.4f}")

In [ ]:
# 8. Flush Drive writes before closing the runtime
drive.flush_and_unmount()
print('Drive flushed. Results folder:', OUT)